# EXPERIMENT: extra features and other models (2-GPU, overnight run). NOT part of the report

Separate experiment: not `housing_part1.ipynb`, results must not be mixed with it. Three independent parts, every run is cross-validated on the **same 10 folds** as the report and compared with the **report's final NN** (k-means 1200/500, `REF_CFG`, CV R² 0.868):

* **Part A — label-free features (from the literature).** Each block is added to the report's feature set, one at a time; a block is kept only if it is *clearly* better (lower CV RMSE **and** better in at least 7 of 10 folds), then the kept blocks are tested together.
  * `log_totals`: log of the skewed district totals (rooms, bedrooms, population, households).
  * `density`: distance (km) to the nearest / 10 nearest districts, number of other districts at the same location.
  * `coast`: distance (km) to the nearest district labelled NEAR OCEAN and NEAR BAY: a proxy for the distance to the coast (the coastal premium is one of the best documented effects on California house prices).
  * `slx`: *spatial lag of the inputs* (spatial econometrics): the neighbours' average income, house age and rooms per household (10 and 50 nearest districts), and my income minus theirs.
  All of these use only the **inputs** of the other training districts, never prices.
* **Part C — other neural networks (label-free, report features)**, plus boosting as a reference: **TabM** (ICLR 2025), **RealMLP** (NeurIPS 2024), **TabPFN v2** (Nature 2025; pretrained transformer, max 500 features, so it gets the k-means 75/15 set), **CatBoost**. Then simple blends (averages) with the report NN. Every model writes a submission file.
* **Part B — neighbour prices: out-of-fold vs leave-one-out (uses the response variable).** Is the old notebook's leave-one-out version better? 2×2 design: out-of-fold / leave-one-out × our columns / the old notebook's columns (k = 1…20 + "comparables").

Parts that fail (e.g. a package that cannot be installed) are skipped with an error message; the rest of the notebook keeps running. Everything is saved to `OUTPUT_DIR = .../outputs_EXPERIMENT_features_models` after each run; a re-run loads it instead of retraining. Kaggle: GPU T4 x2, **internet ON** (Part C installs packages). Expected run time: about 4 hours.

# Packages

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects
import seaborn as sns
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from scipy.spatial.distance import cdist
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, FunctionTransformer, OneHotEncoder
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_validate
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.metrics.pairwise import rbf_kernel
import itertools
import sys, subprocess, traceback
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.neighbors import NearestNeighbors
import time
from tqdm.auto import tqdm

# For NN: Use CUDA if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using acceleration device: {device}")

In [ ]:
# Part C packages (needs internet ON on Kaggle). TabPFN pinned to v2: newer versions download gated weights (Hugging Face login)
PIP = ["tabm==0.0.3", "rtdl_num_embeddings", "pytabkit", "tabpfn==2.0.9", "catboost"]
result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PIP], capture_output=True, text=True)
print("pip install:", "ok" if result.returncode == 0 else "FAILED (Part C models that need these packages will be skipped)\n" + result.stderr[-1500:])

# General settings and data import

In [ ]:
SEED=118

#from google.colab import drive # Only if running from Colab
#drive.mount('/content/drive') # Only if running from Colab
DATA_DIR = "/kaggle/working" # Change to appropriate local path
OUTPUT_DIR = os.path.join(DATA_DIR, "outputs_EXPERIMENT_features_models") # separate from the report outputs
os.makedirs(OUTPUT_DIR, exist_ok=True) # If the folder doesn't exist, create it

x_train_df = pd.read_csv("/kaggle/input/datasets/eduardovitale/california-housing-kaggle-comp/x_train_houses.csv")
y_train_df = pd.read_csv("/kaggle/input/datasets/eduardovitale/california-housing-kaggle-comp/y_train_houses.csv")
x_test_df = pd.read_csv("/kaggle/input/datasets/eduardovitale/california-housing-kaggle-comp/x_test_houses.csv")

features=x_train_df.columns[1:].tolist()
y_tr_raw=y_train_df.iloc[:, 1].values

# IDA and EDA

In [ ]:
x_train = x_train_df[features]
x_test = x_test_df[features]

# 1. Look for missing data
print(x_train.isna().sum())
print(x_test.isna().sum()) # there is missing data in total_bedrooms

# 2. Numerical variables' distributions
print(x_train.describe())
x_train.assign(Price=y_tr_raw).hist(bins=40, figsize=(12,8))
plt.tight_layout()
plt.show()

# 3. Plot house on a map based on latitude/longitude
cities=np.array([[34.1141,-118.4068],   # Los Angeles
                 [38.5677,-121.4685],   # Sacramento
                 [32.8313,-117.1222],   # San Diego
                 [37.7558,-122.4449],   # San Francisco
                 [37.3012,-121.8480]])  # San Jose
# source: https://simplemaps.com/data/us-cities
city_names = ["Los Angeles", "Sacramento", "San Diego", "San Francisco", "San Jose"]

plt.figure(figsize=(8,8))
plt.scatter(x_train["longitude"], x_train["latitude"], c=y_tr_raw, alpha=0.4, s=7)
plt.colorbar(label="Price")
plt.scatter(cities[:, 1], cities[:, 0], color="red", edgecolor="white", s=150, marker="*")
for i, name in enumerate(city_names):
    txt = plt.text(cities[i, 1] + 0.15, cities[i, 0] - 0.05, name, color="black", fontsize=10, fontweight="bold")
    txt.set_path_effects([path_effects.withStroke(linewidth=3, foreground="white")])
plt.gca().set_aspect("equal")
plt.xlabel("longitude")
plt.ylabel("latitude")
plt.show()
# Expensive houses near the big cities. An engineered feature that measures distance from them could have improving effects?

# 4. Mean price by Ocean proximity
sns.boxplot(data=x_train.assign(Price=y_tr_raw), x="ocean_proximity", y="Price",
            order=x_train.assign(Price=y_tr_raw).groupby("ocean_proximity")["Price"].mean().sort_values(ascending=False).index)
plt.show()
# Expensive houses are located near the coast

# 5. Correlations
heatmap = sns.heatmap(x_train.assign(Price=y_tr_raw).select_dtypes("number").corr(), cmap="coolwarm", annot=True)
## Correlations with response variable "Price"
corr_price = x_train.assign(Price=y_tr_raw).select_dtypes("number").corr()["Price"].sort_values(key=abs, ascending=False)
corr_price
# most correlated variable: median_income

# Feature engineering

In [ ]:
# Histograms: the totals measure district size -> per-household ratios describe the typical home
def ratios(d):
    return d.assign(rooms_per_household=d["total_rooms"]/d["households"],
                    population_per_household=d["population"]/d["households"],
                    bedrooms_per_room=d["total_bedrooms"]/d["total_rooms"],
                    #income_per_room=d["median_income"]/d["total_rooms"]
                   )

def new_features(d):
    return d.assign(income_x_age=d["median_income"]*d["housing_median_age"], # Old houses, high income -> possible wealthy area
                    income_per_room=d["median_income"]/d["rooms_per_household"], # income per unit of housing space
                   )

# Lat/long as a smooth 3D position on the sphere, to account for the earth's curvature
def sphere_coords(d):
    lat=np.radians(d["latitude"])
    lon=np.radians(d["longitude"])
    return d.assign(x_coord=np.cos(lat)*np.cos(lon),
                    y_coord=np.cos(lat)*np.sin(lon),
                    z_coord=np.sin(lat))

# Map: expensive houses are near the big cities -> distance to the 2 closest main cities
def dist_city(d):
    dists = cdist(d[["latitude", "longitude"]].to_numpy(), cities)
    sorted_dists = np.sort(dists, axis=1)
    return d.assign(dist_nearest_city=sorted_dists[:, 0],
                    dist_2nd_nearest_city=sorted_dists[:, 1])

# Map: split CA into k-means clusters, and score each district by its similarity (closeness) to each cluster centroid.
# Clusters use coordinates only (no sample weights), so these features contain no price information.
# KM_GAMMA and KM_CLUSTERS were chosen by cross-validation (see the markdown below); they are the default everywhere.
KM_GAMMA, KM_CLUSTERS = 500, 1200

def kmeans_pipeline(gamma=KM_GAMMA, n_clusters=KM_CLUSTERS):
    kmeans = KMeans(n_clusters, n_init=10, random_state=SEED)
    return make_pipeline(kmeans, FunctionTransformer(lambda d: np.exp(-gamma * d ** 2)))

# ---------- EXPERIMENT: extra features ----------
def log_totals(d):
    # Part A: log of the skewed district totals (histograms): compresses the long right tails
    return d.assign(**{f"log_{c}": np.log1p(d[c]) for c in ["total_rooms", "total_bedrooms", "population", "households"]})

def to_km(lat, lon):
    # (latitude, longitude) in km: 1 degree of latitude ~ 111 km, 1 degree of longitude ~ 111 km x cos(latitude)
    return np.column_stack([lat * 111.0, lon * 111.0 * np.cos(np.radians(lat))])

def knn_excluding(index, Z, k, own=None):
    # k nearest points of the fitted `index` for each row of Z. own[i] = position of row i inside the index (-1 if absent):
    # that point (the district itself) is excluded; same-location twins are kept
    if own is None:
        return index.kneighbors(Z, n_neighbors=min(k, index.n_samples_fit_))
    dist, ind = index.kneighbors(Z, n_neighbors=min(k + 1, index.n_samples_fit_))
    order = np.argsort(ind == own[:, None], axis=1, kind="stable") # the district itself goes last
    return np.take_along_axis(dist, order, axis=1)[:, :k], np.take_along_axis(ind, order, axis=1)[:, :k]

CTX_COLS = ["latitude", "longitude", "median_income", "housing_median_age", "rooms_per_household", "ocean_proximity"]
CTX_NUM = ["median_income", "housing_median_age", "rooms_per_household"]

class SpatialContext(BaseEstimator, TransformerMixin):
    # Part A, LABEL-FREE: uses only the inputs of the other training districts, never prices.
    #   "density": distance (km) to the nearest / 10 nearest training districts, number of other districts at the same location
    #   "coast":   distance (km) to the nearest training district labelled NEAR OCEAN, and NEAR BAY (proxy for the coast)
    #   "slx":     neighbours' average income, age, rooms per household (10 and 50 nearest) + my income minus theirs (10 nearest)
    # Training rows (fit_transform): the district itself is excluded. New rows (transform): all training districts.
    def __init__(self, parts=("density",)):
        self.parts = parts

    def fit(self, X, y=None):
        self.xy_ = to_km(X["latitude"].to_numpy(float), X["longitude"].to_numpy(float))
        self.num_ = X[CTX_NUM].to_numpy(float)
        self.index_ = NearestNeighbors().fit(self.xy_)
        cat = X["ocean_proximity"].to_numpy()
        self.coast_ = {}
        for label in ("NEAR OCEAN", "NEAR BAY"):
            mask = cat == label
            pos = np.full(len(cat), -1); pos[mask] = np.arange(mask.sum())
            self.coast_[label] = (NearestNeighbors().fit(self.xy_[mask]), pos)
        return self

    def _features(self, X, own):
        xy = to_km(X["latitude"].to_numpy(float), X["longitude"].to_numpy(float))
        cols = []
        if "density" in self.parts:
            dist, _ = knn_excluding(self.index_, xy, 10, own)
            cols += [dist[:, 0], dist.mean(axis=1), (dist < 1e-6).sum(axis=1)]
        if "coast" in self.parts:
            for label, (index, pos) in self.coast_.items():
                dist, _ = knn_excluding(index, xy, 1, None if own is None else pos[own])
                cols.append(dist[:, 0])
        if "slx" in self.parts:
            _, ind = knn_excluding(self.index_, xy, 50, own)
            nb = self.num_[ind] # (rows, 50 neighbours, 3 inputs)
            cols += [nb[:, :10].mean(axis=1), nb.mean(axis=1), (X["median_income"].to_numpy(float) - nb[:, :10, 0].mean(axis=1))[:, None]]
        return np.column_stack(cols)

    def transform(self, X):
        return self._features(X, own=None)

    def fit_transform(self, X, y=None):
        self.fit(X)
        return self._features(X, own=np.arange(len(X)))

class NeighbourPrice(BaseEstimator, TransformerMixin):
    # Part B, USES THE RESPONSE: mean price of the k nearest TRAINING districts (k in ks), optionally
    #   income adjustment (neighbours' price per unit of income x my income, neighbours' income, my income minus theirs) and
    #   "comparables" (mean price of the 5 / 10 districts nearest in space AND income: 1 unit of income counts as ~5.5 km).
    # Training rows: out-of-fold (default: 5 parts, each from the other 4) or leave-one-out (loo=True: all other training rows,
    # as in the old notebook). New rows: all training rows. Plus the label-free density columns.
    def __init__(self, ks=(5, 10, 20, 50), income=False, comparables=False, loo=False, n_splits=5):
        self.ks, self.income, self.comparables, self.loo, self.n_splits = ks, income, comparables, loo, n_splits

    @staticmethod
    def _space_income(X):
        return np.column_stack([to_km(X[:, 0], X[:, 1]), X[:, 2] * 5.55])

    def _price_cols(self, X_ref, y_ref, X, own=None):
        _, ind = knn_excluding(NearestNeighbors().fit(to_km(X_ref[:, 0], X_ref[:, 1])), to_km(X[:, 0], X[:, 1]), max(self.ks), own)
        p = y_ref[ind]
        cols = [p[:, :k].mean(axis=1) for k in self.ks]
        if self.income:
            inc = X_ref[ind, 2]
            cols += [(p / inc)[:, :k].mean(axis=1) * X[:, 2] for k in (10, 20)]
            cols += [inc[:, :10].mean(axis=1), X[:, 2] - inc[:, :10].mean(axis=1)]
        if self.comparables:
            _, ind3 = knn_excluding(NearestNeighbors().fit(self._space_income(X_ref)), self._space_income(X), 10, own)
            cols += [y_ref[ind3[:, :5]].mean(axis=1), y_ref[ind3].mean(axis=1)]
        return np.column_stack(cols)

    def _density_cols(self, X, own):
        dist, _ = knn_excluding(self.index_, to_km(X[:, 0], X[:, 1]), 10, own)
        return np.column_stack([dist[:, 0], dist.mean(axis=1), (dist < 1e-6).sum(axis=1)])

    def fit(self, X, y):
        self.X_, self.y_ = np.asarray(X, dtype=float), np.asarray(y, dtype=float)
        self.index_ = NearestNeighbors().fit(to_km(self.X_[:, 0], self.X_[:, 1]))
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return np.column_stack([self._price_cols(self.X_, self.y_, X), self._density_cols(X, None)])

    def fit_transform(self, X, y):
        self.fit(X, y)
        X, y, own = self.X_, self.y_, np.arange(len(self.X_))
        if self.loo:
            prices = self._price_cols(X, y, X, own)
        else:
            prices = None
            for tr, va in KFold(self.n_splits, shuffle=True, random_state=SEED).split(X):
                cols = self._price_cols(X[tr], y[tr], X[va])
                if prices is None:
                    prices = np.empty((len(X), cols.shape[1]))
                prices[va] = cols
        return np.column_stack([prices, self._density_cols(X, own)])

# Model utilities (same as the report notebook, plus the experimental options)

In [ ]:
cv=KFold(10, shuffle=True, random_state=SEED) # same folds for every model -> paired comparisons
scoring={"mse":"neg_mean_squared_error", "mae":"neg_mean_absolute_error", "r2":"r2"}

# steps: row-wise feature functions, applied first (they only use their own row, so no leakage).
# Everything that is fitted (imputer, scalers, ridge) lives inside the pipeline,
# so cross_validate refits it on each training fold only.
# EXPERIMENT: kmeans = False / True (report behaviour) or "+"-separated options:
#   "km" (k-means similarities), "density" / "coast" / "slx" (Part A, SpatialContext),
#   "nb" (Part B, NeighbourPrice) with "inc" (income adjustment), "comp" (comparables), "loo" (leave-one-out), "aiks" (k = 1, 3, 5, 10, 20)
def location_parts(kmeans, gamma, n_clusters):
    opts = set({False: "", True: "km"}.get(kmeans, kmeans).split("+"))
    parts, reused = [], set()
    if "km" in opts:
        parts.append(("km", kmeans_pipeline(gamma, n_clusters), ["latitude", "longitude"]))
    ctx = tuple(p for p in ("density", "coast", "slx") if p in opts)
    if ctx:
        parts.append(("ctx", make_pipeline(SpatialContext(ctx), StandardScaler()), CTX_COLS))
        reused |= set(CTX_NUM)
    if "nb" in opts:
        np_ = NeighbourPrice(ks=(1, 3, 5, 10, 20) if "aiks" in opts else (5, 10, 20, 50), income="inc" in opts,
                             comparables="comp" in opts, loo="loo" in opts)
        parts.append(("nb", make_pipeline(np_, StandardScaler()), ["latitude", "longitude", "median_income"]))
        reused |= {"median_income"}
    if reused: # columns used by these steps would disappear from the remainder: pass them on as normal features as well
        parts.append(("reused", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), sorted(reused)))
    return parts

def build(steps, model, degree=1, kmeans=False, gamma=KM_GAMMA, n_clusters=KM_CLUSTERS):
    num_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
    cat_pipeline = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    preprocessing = ColumnTransformer(
        [("cat", cat_pipeline, make_column_selector(dtype_exclude="number"))] +
        location_parts(kmeans, gamma, n_clusters),
        remainder=num_pipeline)
    return make_pipeline(*[FunctionTransformer(f) for f in steps], preprocessing, model)

def cross_val_all(models, n_jobs=-1):
    return {name:cross_validate(m, x_train, y_tr_raw, cv=cv, scoring=scoring, return_train_score=True, n_jobs=n_jobs)
            for name, m in models.items()}

# train_* = in-sample, cv_* = out-of-sample (mean over the 10 folds).
# rmse_gain: positive = better than the reference row. folds_better: in how many of the 10 folds it beat the reference.
# Reference = the row above, or the first row if vs_first=True (the first row compares with itself).
def summarise(res, vs_first=False):
    names=list(res)
    mse={n:-res[n]["test_mse"] for n in names}
    out=pd.DataFrame(index=names)
    out["train_rmse"]=[np.sqrt(-res[n]["train_mse"].mean()) for n in names]
    out["train_r2"]=[res[n]["train_r2"].mean() for n in names]
    out["cv_rmse"]=[np.sqrt(mse[n].mean()) for n in names]
    out["cv_mae"]=[-res[n]["test_mae"].mean() for n in names]
    out["cv_r2"]=[res[n]["test_r2"].mean() for n in names]
    ref=[names[0] if vs_first else names[max(i-1, 0)] for i in range(len(names))]
    out["rmse_gain"]=[out.loc[r, "cv_rmse"] - out.loc[n, "cv_rmse"] for n, r in zip(names,ref)]
    out["folds_better"]=[int((mse[n] < mse[r]).sum()) for n, r in zip(names, ref)]
    return out.round(3)

In [ ]:
keep_steps=[ratios, new_features, dist_city, sphere_coords]

feats = {"report": (keep_steps, "km"),                          # the report's final feature set
         # Part A (label-free)
         "report+log_totals": (keep_steps + [log_totals], "km"),
         "report+density": (keep_steps, "km+density"),
         "report+coast": (keep_steps, "km+coast"),
         "report+slx": (keep_steps, "km+slx"),
         # Part B (neighbour prices, use the response)
         "nb_inc_oof": (keep_steps, "nb+inc"),                   # our version (out-of-fold)
         "nb_inc_loo": (keep_steps, "nb+inc+loo"),               # same columns, leave-one-out
         "nb_old_oof": (keep_steps, "nb+inc+comp+aiks"),         # the old notebook's columns, out-of-fold
         "nb_old_loo": (keep_steps, "nb+inc+comp+aiks+loo")}     # the old notebook's columns, leave-one-out (as it was)

# Neural network

In [ ]:
# Utility functions
def to_tensor(array, dev=None):
    return torch.tensor(np.asarray(array), dtype=torch.float32, device=dev or device)

def parse_architecture(arch_str):
    # converts a string like "128-64" into a tuple (128, 64)
    if arch_str == "linear":
        return ()
    return tuple(int(w) for w in arch_str.split("-"))

# Preprocessing (imputer, scalers, k-means, one-hot) is fitted on the fitting rows ONLY,
# then applied to every other set (early-stopping, validation and test rows)
def prep_fit(steps, X_fit, y_fit, *other_splits, kmeans=False, gamma=KM_GAMMA, n_clusters=KM_CLUSTERS):
    preprocessor = build(steps, "passthrough", kmeans=kmeans, gamma=gamma, n_clusters=n_clusters)
    X_fit_transformed = preprocessor.fit_transform(X_fit, y_fit)
    other_transformed = [preprocessor.transform(X) for X in other_splits]
    return [X_fit_transformed] + other_transformed

# 1. Construct the network: made it flexible to allow testing for multiple architectures
def build_network(n_inputs, hidden_widths, activation, dropout, use_batchnorm=False):
    # hidden_widths=() gives linear regression
    layers=[]
    in_features=n_inputs

    for width in hidden_widths:
        layers.append(nn.Linear(in_features, width))
        if use_batchnorm:
            layers.append(nn.BatchNorm1d(width))
        layers.append(ACTIVATIONS[activation]())
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        in_features=width

    layers.append(nn.Linear(in_features, 1))
    return nn.Sequential(*layers)

# 2. Train the network
def train_network(X_train, y_train, X_val, y_val, hidden_widths=(128, 64), activation="relu",
                  learning_rate=1e-3, dropout=0.1, weight_decay=1e-4, batch_size=256, use_batchnorm=False,
                  seed=SEED, max_epochs=300, patience=30, device_name=None):
    # Train with MSE loss and AdamW. LR halves on plateau; stops early on validation RMSE and restores the best epoch's weights
    torch.manual_seed(seed)
    dev = torch.device(device_name) if device_name else device # [2-GPU] the GPU this network trains on

    # standardize the target using training statistics only
    y_mean, y_std = y_train.mean(), y_train.std()
    X_train_t = to_tensor(X_train, dev)
    X_val_t = to_tensor(X_val, dev)
    y_train_t = to_tensor((y_train-y_mean)/y_std, dev).unsqueeze(1)

    net = build_network(X_train_t.shape[1], hidden_widths, activation, dropout, use_batchnorm).to(dev)
    optimizer = optim.AdamW(net.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=10)
    loss_fn = nn.MSELoss()

    def compute_rmse(X, y_true):
        # RMSE in dollars, with dropout/batchnorm turned off
        net.eval()
        with torch.no_grad():
            preds = net(X).squeeze(1).cpu().numpy() * y_std + y_mean
        return np.sqrt(np.mean((preds - y_true) ** 2))

    best_val_rmse = np.inf
    best_epoch = 0
    best_weights = None
    history = []

    for epoch in range(max_epochs):
        net.train()
        shuffled_idx = torch.randperm(len(X_train_t), device=dev)

        for start in range(0, len(shuffled_idx), batch_size):
            batch_idx = shuffled_idx[start:start + batch_size]
            if len(batch_idx) < 2:
                continue  # batchnorm needs at least 2 rows
            optimizer.zero_grad()
            loss = loss_fn(net(X_train_t[batch_idx]), y_train_t[batch_idx])
            loss.backward()
            optimizer.step()

        train_rmse = compute_rmse(X_train_t, y_train)
        val_rmse = compute_rmse(X_val_t, y_val)
        history.append((train_rmse, val_rmse))
        scheduler.step(val_rmse)

        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_epoch = epoch
            best_weights = {k: v.clone() for k, v in net.state_dict().items()}
        elif epoch - best_epoch >= patience:
            break

    net.load_state_dict(best_weights)
    return {"net": net, "y_mean": y_mean, "y_std": y_std, "best_epoch": best_epoch, "history": history}

# 3. Use the network to make predictions
def predict(result, X):
    result["net"].eval()
    dev = next(result["net"].parameters()).device # [2-GPU] predict on the network's own GPU
    with torch.no_grad():
        preds = result["net"](to_tensor(X, dev)).squeeze(1).cpu().numpy()
    return preds * result["y_std"] + result["y_mean"]

# 4. Tuning
def run_grid(configs, feature_set_name):
    # [2-GPU] same networks as the 1-GPU version, trained in parallel by train_many (one worker per GPU)
    X_train, X_val = data[feature_set_name]
    n_train = len(ya)
    arrays = {"X": np.vstack([X_train, X_val]).astype(np.float32), "y": np.concatenate([ya, yb])}
    fit, stop = np.arange(n_train), np.arange(n_train, n_train + len(yb))
    jobs = [dict(X="X", y="y", fit=fit, stop=stop, predict=[("X", fit), ("X", stop)], cfg=config) for config in configs]
    results_rows = []
    histories = []

    for config, res in zip(configs, train_many(arrays, jobs)):
        pred_train, pred_val = res["preds"]
        arch_label = "-".join(map(str, config["hidden_widths"])) or "linear"
        other_params = {k: v for k, v in config.items() if k != "hidden_widths"}
        results_rows.append({
            "feat": feature_set_name,
            "arch": arch_label,
            **other_params,
            "best_epoch": res["best_epoch"] + 1,
            "train_rmse": np.sqrt(mean_squared_error(ya, pred_train)),
            "val_rmse": np.sqrt(mean_squared_error(yb, pred_val)),
            "val_mae": mean_absolute_error(yb, pred_val),
            "val_r2": r2_score(yb, pred_val),
            "secs": res["secs"],
        })
        histories.append(res["history"])
    return pd.DataFrame(results_rows), histories

# For visualization purposes:
def show_grid(df, rows, cols, value="val_rmse"):
    # Pivot into a rows x cols table of validation RMSE, colored (darker=lower error)
    return df.pivot_table(index=rows, columns=cols, values=value).round(0).style.background_gradient(cmap="viridis_r", axis=None)

def plot_curves(histories, titles):
    # Diagnostic plot for convergence
    n_plots = len(histories)
    n_cols = 4
    n_rows = int(np.ceil(n_plots / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows), sharey=True)
    axes = np.atleast_1d(axes).ravel()

    lowest_val_rmse = min(min(val for _, val in hist) for hist in histories)

    for ax, hist, title in zip(axes, histories, titles):
        hist = np.array(hist)
        ax.plot(hist[:, 0], label="train")
        ax.plot(hist[:, 1], label="validation")
        ax.set_title(title)
        ax.set_ylim(lowest_val_rmse * 0.6, lowest_val_rmse * 2)

    # hide unused subplot slots
    for ax in axes[n_plots:]:
        ax.axis("off")

    axes[0].legend()
    fig.supxlabel("epoch")
    fig.supylabel("RMSE ($)")
    plt.tight_layout()
    plt.show()

In [ ]:
# [2-GPU] Train several networks at the same time: one worker PROCESS per GPU.
# Only the order of the work changes: every network gets exactly the same rows, configuration and seed as in
# housing_part1.ipynb, so the results are the same (up to the usual GPU rounding differences).
# Processes, not threads: each process has its own random-number generators, so a network's seed controls only that network.
# A job = one network: row indices (fit / stop) into a matrix of `arrays`, the configuration (incl. seed),
# and the matrices (or rows) to predict. Large arrays are shared with the workers through files (joblib memmapping).
from joblib import Parallel, delayed

DEVICES = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cpu"]
print("networks are trained on:", DEVICES)

def _train_chunk(arrays, jobs, device_name):
    # runs inside one worker: trains its jobs one after the other on one device
    out = []
    for job in jobs:
        start_time = time.time()
        X, y = arrays[job["X"]], arrays[job["y"]]
        result = train_network(X[job["fit"]], y[job["fit"]], X[job["stop"]], y[job["stop"]], device_name=device_name, **job["cfg"])
        preds = [predict(result, arrays[name] if rows is None else arrays[name][rows]) for name, rows in job["predict"]]
        out.append(dict(preds=preds, best_epoch=result["best_epoch"], history=result["history"], secs=time.time() - start_time))
    return out

def train_many(arrays, jobs):
    # splits the jobs over the devices (job i -> device i % n_devices) and returns one result per job, in the original order
    chunks = [list(range(d, len(jobs), len(DEVICES))) for d in range(len(DEVICES))]
    if len(DEVICES) == 1:
        parts = [_train_chunk(arrays, jobs, DEVICES[0])]
    else:
        parts = Parallel(n_jobs=len(DEVICES))(delayed(_train_chunk)(arrays, [jobs[i] for i in chunk], dev)
                                              for chunk, dev in zip(chunks, DEVICES))
    results = [None] * len(jobs)
    for chunk, part in zip(chunks, parts):
        for i, res in zip(chunk, part):
            results[i] = res
    return results

In [ ]:
def cross_validate_nn(cfg, feat, seeds=(89, 233, 1597)):
    # Honest CV: each fold trains a fresh ensemble of `seeds` networks, with preprocessing fit on that
    # fold's training rows only. Early stopping watches a held-out 10% split (`stop`), never the
    # validation fold (`val`), so nothing about the validation fold leaks into training decisions.
    # Keys match cross_validate's naming (train_mse/test_mse/...) so summarise() works on both.
    # [2-GPU] the preprocessing of all folds is done first, then all 10 x 3 networks are trained in parallel.
    steps, km = feats[feat]
    y = y_tr_raw
    fold_scores = dict(train_mse=[], test_mse=[], test_mae=[], train_r2=[], test_r2=[])
    test_preds = []
    oof_preds = np.zeros(len(y))

    arrays, jobs, folds = {}, [], []
    for f, (train_idx, val_idx) in enumerate(tqdm(list(cv.split(x_train)), desc="preprocessing folds")):
        fit_idx, stop_idx = train_test_split(train_idx, test_size=0.1, random_state=SEED)
        X_fit, X_stop, X_val, X_test = prep_fit(
            steps, x_train.iloc[fit_idx], y[fit_idx], x_train.iloc[stop_idx],
            x_train.iloc[val_idx], x_test, kmeans=km)
        # rows of fold f stacked as [fit | stop | val]; the test set in its own matrix
        arrays[f"X{f}"] = np.vstack([X_fit, X_stop, X_val]).astype(np.float32)
        arrays[f"y{f}"] = np.concatenate([y[fit_idx], y[stop_idx], y[val_idx]])
        arrays[f"T{f}"] = np.asarray(X_test, dtype=np.float32)
        n_fit, n_stop = len(fit_idx), len(stop_idx)
        r_fit, r_stop, r_val = np.arange(n_fit), np.arange(n_fit, n_fit + n_stop), np.arange(n_fit + n_stop, n_fit + n_stop + len(val_idx))
        for seed in seeds:
            jobs.append(dict(X=f"X{f}", y=f"y{f}", fit=r_fit, stop=r_stop,
                             predict=[(f"X{f}", r_fit), (f"X{f}", r_val), (f"T{f}", None)], cfg={**cfg, "seed": seed}))
        folds.append((fit_idx, val_idx))

    results = train_many(arrays, jobs)

    for f, (fit_idx, val_idx) in enumerate(folds):
        fold_results = results[f * len(seeds):(f + 1) * len(seeds)]
        fit_pred = np.mean([r["preds"][0] for r in fold_results], axis=0)  # ensemble = average of the seeds
        val_pred = np.mean([r["preds"][1] for r in fold_results], axis=0)
        test_preds.append(np.mean([r["preds"][2] for r in fold_results], axis=0))
        oof_preds[val_idx] = val_pred

        fold_scores["train_mse"].append(-mean_squared_error(y[fit_idx], fit_pred))
        fold_scores["test_mse"].append(-mean_squared_error(y[val_idx], val_pred))
        fold_scores["test_mae"].append(-mean_absolute_error(y[val_idx], val_pred))
        fold_scores["train_r2"].append(r2_score(y[fit_idx], fit_pred))
        fold_scores["test_r2"].append(r2_score(y[val_idx], val_pred))

    fold_scores = {k: np.array(v) for k, v in fold_scores.items()}
    return fold_scores, np.mean(test_preds, axis=0), oof_preds

In [ ]:
# Features to test
ACTIVATIONS = {"relu":nn.ReLU, "leaky_relu":nn.LeakyReLU, "elu":nn.ELU, "silu":nn.SiLU, "gelu":nn.GELU, "tanh":nn.Tanh}
# REF_CFG   = the report's final NN configuration (Parts A and C)
# START_CFG = the configuration used for the neighbour-price feature sets (Part B, as in the neighbour-price experiment)
REF_CFG = dict(hidden_widths=(256,128,64), activation="leaky_relu", learning_rate=0.0005, dropout=0.2,
               weight_decay=0.0, batch_size=256, use_batchnorm=True)
START_CFG = dict(hidden_widths=(256,128), activation="relu", learning_rate=0.005, dropout=0.2,
                 weight_decay=0.001, batch_size=128, use_batchnorm=True)

# Saving / loading runs, blends

In [ ]:
RES_DIR = os.path.join(OUTPUT_DIR, "runs")
os.makedirs(RES_DIR, exist_ok=True)
METRICS = ["train_mse", "test_mse", "test_mae", "train_r2", "test_r2"]

def save_run(name, run):
    pd.DataFrame({m: run["scores"][m] for m in METRICS}).to_csv(os.path.join(RES_DIR, f"{name}.csv"), index_label="fold")
    np.save(os.path.join(RES_DIR, f"{name}_oof.npy"), run["oof"])
    np.save(os.path.join(RES_DIR, f"{name}_test.npy"), run["test"])

def load_run(name):
    path = os.path.join(RES_DIR, f"{name}.csv")
    if not os.path.exists(path):
        return None
    d = pd.read_csv(path)
    return dict(scores={m: d[m].to_numpy() for m in METRICS},
                oof=np.load(os.path.join(RES_DIR, f"{name}_oof.npy")), test=np.load(os.path.join(RES_DIR, f"{name}_test.npy")))

def nn_run(name, cfg, feat):
    # cross_validate_nn (30 networks: 10 folds x 3 seeds) with saving; a re-run loads the saved result
    run = load_run(name)
    if run is None:
        start_time = time.time()
        scores, test, oof = cross_validate_nn(cfg, feat)
        run = dict(scores=scores, oof=oof, test=test)
        save_run(name, run)
        print(f"{name}: cv_rmse={np.sqrt(-np.mean(scores['test_mse'])):,.0f} ({time.time() - start_time:.0f}s)")
    return run

def clearly_better(new, old):
    # lower CV RMSE and better in at least 7 of the 10 folds
    return (-new["test_mse"].mean() < -old["test_mse"].mean()) and ((new["test_mse"] > old["test_mse"]).sum() >= 7)

def blend(runs):
    # simple average of the out-of-fold and test predictions of several runs, scored on the same validation folds
    oof = np.mean([r["oof"] for r in runs], axis=0)
    scores = {m: [] for m in METRICS}
    for train_idx, val_idx in cv.split(x_train):
        y_val, p_val = y_tr_raw[val_idx], oof[val_idx]
        scores["train_mse"].append(np.nan); scores["train_r2"].append(np.nan) # no single training set for a blend
        scores["test_mse"].append(-mean_squared_error(y_val, p_val))
        scores["test_mae"].append(-mean_absolute_error(y_val, p_val))
        scores["test_r2"].append(r2_score(y_val, p_val))
    return dict(scores={m: np.array(v) for m, v in scores.items()}, oof=oof, test=np.mean([r["test"] for r in runs], axis=0))

def save_submission(test_pred, filename):
    pred = np.clip(test_pred, y_tr_raw.min(), y_tr_raw.max())
    pd.DataFrame({"ID": x_test_df.iloc[:, 0], "Price": pred}).to_csv(os.path.join(OUTPUT_DIR, filename), index=False)

# Part A: label-free features

Each block is added to the report's feature set (k-means 1200/500 + engineered features), same NN configuration (`REF_CFG`) and folds. Kept only if clearly better; then the kept blocks together (confirmation).

In [ ]:
ref = nn_run("A_report", REF_CFG, "report")
A_BLOCKS = ["log_totals", "density", "coast", "slx"]
a_runs = {b: nn_run(f"A_report+{b}", REF_CFG, f"report+{b}") for b in A_BLOCKS}
display(summarise({"report (k-means 1200/500)": ref["scores"], **{f"+ {b}": r["scores"] for b, r in a_runs.items()}}, vs_first=True))

KEEP_A = None # EDIT: list of blocks to override the automatic choice
kept = KEEP_A if KEEP_A is not None else [b for b, r in a_runs.items() if clearly_better(r["scores"], ref["scores"])]
print("clearly useful blocks:", kept)
if len(kept) > 1:
    steps = keep_steps + ([log_totals] if "log_totals" in kept else [])
    opts = "+".join(["km"] + [b for b in kept if b != "log_totals"])
    feats["report+A"] = (steps, opts)
    a_all = nn_run("A_report+" + "+".join(kept), REF_CFG, "report+A")
    display(summarise({"report": ref["scores"], "+ " + ", ".join(kept): a_all["scores"]}, vs_first=True))
elif len(kept) == 1:
    a_all = a_runs[kept[0]]
if kept:
    save_submission(a_all["test"], "submission_EXP_A_features.csv")

# Part C: other neural networks (label-free, report features) + boosting reference + blends

Same folds and the same early-stopping split as the report NN (the models that use one early-stop on the 10% `stop` rows). One model per fold (TabM already averages 32 internal members). TabPFN gets the k-means **75/15** feature set (~96 columns), because it accepts at most 500 features.

In [ ]:
TABM_EPOCHS, TABM_PATIENCE = 300, 20
REALMLP_KW = {}         # extra arguments for RealMLP_TD_Regressor (defaults of the paper)
CATBOOST_ITERS = 3000
_PREP_CACHE = {}        # preprocessed folds, shared by all Part C models (the k-means fit is the slow part)

def fold_data(feat, f, fit_idx, stop_idx, val_idx, **prep_kwargs):
    key = (feat, f, tuple(sorted(prep_kwargs.items())))
    if key not in _PREP_CACHE:
        steps, km = feats[feat]
        _PREP_CACHE[key] = [np.asarray(a, dtype=np.float32) for a in prep_fit(
            steps, x_train.iloc[fit_idx], y_tr_raw[fit_idx], x_train.iloc[stop_idx], x_train.iloc[val_idx], x_test,
            kmeans=km, **prep_kwargs)]
    return _PREP_CACHE[key]

def cv_model(name, feat, fit_predict, seeds=(89,), **prep_kwargs):
    # fit_predict(X_fit, y_fit, X_stop, y_stop, [X_fit, X_val, X_test], seed) -> list of 3 prediction arrays
    run = load_run(name)
    if run is not None:
        return run
    start_time = time.time()
    y = y_tr_raw
    scores = {m: [] for m in METRICS}
    oof, test_preds = np.zeros(len(y)), []
    for f, (train_idx, val_idx) in enumerate(tqdm(list(cv.split(x_train)), desc=name)):
        fit_idx, stop_idx = train_test_split(train_idx, test_size=0.1, random_state=SEED)
        X_fit, X_stop, X_val, X_test = fold_data(feat, f, fit_idx, stop_idx, val_idx, **prep_kwargs)
        preds = [fit_predict(X_fit, y[fit_idx], X_stop, y[stop_idx], [X_fit, X_val, X_test], s) for s in seeds]
        fit_pred, val_pred, test_pred = [np.mean([p[i] for p in preds], axis=0) for i in range(3)]
        oof[val_idx] = val_pred
        test_preds.append(test_pred)
        scores["train_mse"].append(-mean_squared_error(y[fit_idx], fit_pred))
        scores["test_mse"].append(-mean_squared_error(y[val_idx], val_pred))
        scores["test_mae"].append(-mean_absolute_error(y[val_idx], val_pred))
        scores["train_r2"].append(r2_score(y[fit_idx], fit_pred))
        scores["test_r2"].append(r2_score(y[val_idx], val_pred))
    run = dict(scores={m: np.array(v) for m, v in scores.items()}, oof=oof, test=np.mean(test_preds, axis=0))
    save_run(name, run)
    print(f"{name}: cv_rmse={np.sqrt(-np.mean(run['scores']['test_mse'])):,.0f} ({time.time() - start_time:.0f}s)")
    return run

def tabm_fp(X_fit, y_fit, X_stop, y_stop, X_pred, seed):
    # TabM (MLP + parameter-efficient ensemble of 32 members); MSE of every member, early stopping on the stop rows
    from tabm import TabM
    torch.manual_seed(seed)
    dev = torch.device(DEVICES[0])
    mu, sd = y_fit.mean(), y_fit.std()
    net = TabM.make(n_num_features=X_fit.shape[1], d_out=1).to(dev)
    opt = optim.AdamW(net.parameters(), lr=2e-3, weight_decay=3e-4)
    X_t, y_t, X_s = to_tensor(X_fit, dev), to_tensor((y_fit - mu) / sd, dev), to_tensor(X_stop, dev)

    def predict_std(X):  # average of the 32 members, standardised scale
        net.eval()
        with torch.no_grad():
            return torch.cat([net(xb).mean(1).squeeze(-1) for xb in torch.split(X, 4096)]).cpu().numpy()

    best, best_state, bad = np.inf, None, 0
    for epoch in range(TABM_EPOCHS):
        net.train()
        for idx in torch.randperm(len(X_t), device=dev).split(256):
            opt.zero_grad()
            loss = ((net(X_t[idx]).squeeze(-1) - y_t[idx, None]) ** 2).mean()
            loss.backward()
            opt.step()
        rmse = np.sqrt(np.mean((predict_std(X_s) * sd + mu - y_stop) ** 2))
        if rmse < best:
            best, bad, best_state = rmse, 0, {k: v.clone() for k, v in net.state_dict().items()}
        else:
            bad += 1
            if bad >= TABM_PATIENCE:
                break
    net.load_state_dict(best_state)
    return [predict_std(to_tensor(X, dev)) * sd + mu for X in X_pred]

def realmlp_fp(X_fit, y_fit, X_stop, y_stop, X_pred, seed):
    from pytabkit import RealMLP_TD_Regressor
    m = RealMLP_TD_Regressor(device=DEVICES[0], random_state=seed, verbosity=0, **REALMLP_KW)
    m.fit(X_fit, y_fit, X_val=X_stop, y_val=y_stop)
    return [m.predict(X) for X in X_pred]

def catboost_fp(X_fit, y_fit, X_stop, y_stop, X_pred, seed):
    from catboost import CatBoostRegressor
    m = CatBoostRegressor(iterations=CATBOOST_ITERS, learning_rate=0.05, depth=6, early_stopping_rounds=100,
                          random_seed=seed, verbose=0, task_type="GPU" if DEVICES[0].startswith("cuda") else "CPU")
    m.fit(X_fit, y_fit, eval_set=(X_stop, y_stop))
    return [m.predict(X) for X in X_pred]

def tabpfn_fp(X_fit, y_fit, X_stop, y_stop, X_pred, seed):
    # pretrained transformer: no training, the training rows are its context (no early stopping, so fit + stop rows)
    from tabpfn import TabPFNRegressor
    m = TabPFNRegressor(device=DEVICES[0], ignore_pretraining_limits=True, random_state=seed)
    m.fit(np.vstack([X_fit, X_stop]), np.concatenate([y_fit, y_stop]))
    return [m.predict(X) for X in X_pred]

C_MODELS = {"TabM": ("report", tabm_fp, {}),
            "RealMLP": ("report", realmlp_fp, {}),
            "CatBoost": ("report", catboost_fp, {}),
            "TabPFN_km75": ("report", tabpfn_fp, dict(gamma=15, n_clusters=75))}
c_runs = {"NN (report)": ref}
for name, (feat, fp, kw) in C_MODELS.items():
    try:
        c_runs[name] = cv_model(f"C_{name}", feat, fp, **kw)
        save_submission(c_runs[name]["test"], f"submission_EXP_C_{name}.csv")
    except Exception:
        print(f"{name} FAILED, skipped:")
        traceback.print_exc()

# blends: report NN + each model, and all models together
blends = {f"blend NN + {n}": blend([ref, r]) for n, r in c_runs.items() if n != "NN (report)"}
if len(c_runs) > 2:
    blends["blend all"] = blend(list(c_runs.values()))
for n, r in blends.items():
    save_submission(r["test"], "submission_EXP_C_" + n.replace(" ", "_").replace("+", "and") + ".csv")
display(summarise({**{n: r["scores"] for n, r in c_runs.items()}, **{n: r["scores"] for n, r in blends.items()}}, vs_first=True))

# Part B: neighbour prices, out-of-fold vs leave-one-out (uses the response variable)

NN with `START_CFG` (as in the neighbour-price experiment, where "neighbour prices + income adjustment" reached CV RMSE 41,202). 2×2 design:

| | our columns (k = 5…50, income adjustment) | the old notebook's columns (k = 1…20, income adjustment, comparables) |
|---|---|---|
| out-of-fold | `nb_inc_oof` | `nb_old_oof` |
| leave-one-out | `nb_inc_loo` | `nb_old_loo` (as in the old notebook) |

Note: the CV stays honest in every case, because the **validation** rows always get their neighbours from the fit rows only; leave-one-out only changes how the **training** rows' features are built. So this compares which training scheme generalises better.

In [ ]:
b_runs = {}
for name in ["nb_inc_oof", "nb_inc_loo", "nb_old_oof", "nb_old_loo"]:
    try:
        b_runs[name] = nn_run(f"B_{name}", START_CFG, name)
    except Exception:
        print(f"{name} FAILED, skipped:")
        traceback.print_exc()
display(summarise({"NN report (label-free, REF_CFG)": ref["scores"], **{n: r["scores"] for n, r in b_runs.items()}}, vs_first=True))
for n, r in b_runs.items():
    save_submission(r["test"], f"submission_EXP_B_{n}.csv")